# 通用模型数据生成（宿主机）

支持的模型：`mlp` (784→128→10) / `lenet5` (Conv+Pool+FC)

修改下方 `ARCH` 即可切换模型，其余流程完全一致。

In [1]:
import sys
sys.path.insert(0, '.')
from model_zoo import *

## 1. 选择模型架构

In [9]:
# ===== 在这里选择模型 =====
ARCH = 'mlp'   # 'mlp' or 'lenet5'
# ==========================

SEED = 42
EPOCHS = {'mlp': 10, 'lenet5': 15}[ARCH]
NUM_TEST = 200
OUTPUT_DIR = f'{ARCH}_data'
MODEL_PATH = f'{ARCH}.pt'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Arch: {ARCH}, Device: {device}')

Arch: mlp, Device: cuda


## 2. 训练

In [10]:
model = build_model(ARCH)
float_acc = train(model, epochs=EPOCHS, device=device, seed=SEED)
torch.save(model.state_dict(), MODEL_PATH)
print(f'Saved to {MODEL_PATH}')

Training mlp for 10 epochs (seed=42)
  Epoch 1/10: loss=0.4141, acc=93.34%
  Epoch 2/10: loss=0.1960, acc=95.32%
  Epoch 3/10: loss=0.1418, acc=96.20%
  Epoch 4/10: loss=0.1084, acc=96.53%
  Epoch 5/10: loss=0.0877, acc=97.05%
  Epoch 6/10: loss=0.0732, acc=97.29%
  Epoch 7/10: loss=0.0620, acc=97.38%
  Epoch 8/10: loss=0.0517, acc=97.65%
  Epoch 9/10: loss=0.0457, acc=97.57%
  Epoch 10/10: loss=0.0388, acc=97.73%
Saved to mlp.pt


In [11]:
# 或加载已有模型
# model = build_model(ARCH)
# model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
# model = model.to(device).eval()

## 3. 量化

In [12]:
qparams = quantize(model.to(device).eval(), device)


Quantizing mlp...
  fc1: s_w=0.006430, s_out=0.114035, mult=14, shift=16
  fc2: s_w=0.007374, s_out=0.319246, mult=173, shift=16


## 4. INT8 准确率验证

In [14]:
transform_raw = transforms.ToTensor()
test_set = datasets.MNIST('./data', train=False, download=True, transform=transform_raw)

correct_i8, correct_fp, total = 0, 0, len(test_set)
for i in range(total):
    img, label = test_set[i]
    img_u8 = np.clip(np.round(img.numpy().flatten() * 255), 0, 255).astype(np.uint8)
    pred_i8, _, _ = int8_infer(img_u8, qparams)
    if pred_i8 == label: correct_i8 += 1
    with torch.no_grad():
        pred_fp = model(img.unsqueeze(0).to(device)).argmax(1).item()
    if pred_fp == label: correct_fp += 1
    if i%100 == 0 : print(f"now, epoch {i} in {total}.")

print(f'Float32: {100.*correct_fp/total:.2f}% ({correct_fp}/{total})')
print(f'INT8:    {100.*correct_i8/total:.2f}% ({correct_i8}/{total})')
print(f'Drop:    {100.*(correct_fp-correct_i8)/total:.2f}%')

now, epoch 0 in 10000.
now, epoch 100 in 10000.
now, epoch 200 in 10000.
now, epoch 300 in 10000.
now, epoch 400 in 10000.
now, epoch 500 in 10000.
now, epoch 600 in 10000.
now, epoch 700 in 10000.
now, epoch 800 in 10000.
now, epoch 900 in 10000.
now, epoch 1000 in 10000.
now, epoch 1100 in 10000.
now, epoch 1200 in 10000.
now, epoch 1300 in 10000.
now, epoch 1400 in 10000.
now, epoch 1500 in 10000.
now, epoch 1600 in 10000.
now, epoch 1700 in 10000.
now, epoch 1800 in 10000.
now, epoch 1900 in 10000.
now, epoch 2000 in 10000.
now, epoch 2100 in 10000.
now, epoch 2200 in 10000.
now, epoch 2300 in 10000.
now, epoch 2400 in 10000.
now, epoch 2500 in 10000.
now, epoch 2600 in 10000.
now, epoch 2700 in 10000.
now, epoch 2800 in 10000.
now, epoch 2900 in 10000.
now, epoch 3000 in 10000.
now, epoch 3100 in 10000.
now, epoch 3200 in 10000.
now, epoch 3300 in 10000.
now, epoch 3400 in 10000.
now, epoch 3500 in 10000.
now, epoch 3600 in 10000.
now, epoch 3700 in 10000.
now, epoch 3800 in 10000

## 5. 导出 Hex

In [15]:
test_images = []
test_labels = []
for i in range(NUM_TEST):
    img, label = test_set[i]
    test_images.append(np.clip(np.round(img.numpy().flatten()*255), 0, 255).astype(np.uint8))
    test_labels.append(int(label))

export_hex(qparams, OUTPUT_DIR, test_images, test_labels, NUM_TEST)

  [0000] label=7, pred=7 ✓
  [0001] label=2, pred=2 ✓
  [0002] label=1, pred=1 ✓
  [0003] label=0, pred=0 ✓
  [0004] label=4, pred=4 ✓
  [0005] label=1, pred=1 ✓
  [0006] label=4, pred=4 ✓
  [0007] label=9, pred=9 ✓
  [0008] label=5, pred=5 ✓
  [0009] label=9, pred=9 ✓
  [0010] label=0, pred=0 ✓
  [0011] label=6, pred=6 ✓
  [0012] label=9, pred=9 ✓
  [0013] label=0, pred=0 ✓
  [0014] label=1, pred=1 ✓
  [0015] label=5, pred=5 ✓
  [0016] label=9, pred=9 ✓
  [0017] label=7, pred=7 ✓
  [0018] label=3, pred=3 ✓
  [0019] label=4, pred=4 ✓
  [0020] label=9, pred=9 ✓
  [0021] label=6, pred=6 ✓
  [0022] label=6, pred=6 ✓
  [0023] label=5, pred=5 ✓
  [0024] label=4, pred=4 ✓
  [0025] label=0, pred=0 ✓
  [0026] label=7, pred=7 ✓
  [0027] label=4, pred=4 ✓
  [0028] label=0, pred=0 ✓
  [0029] label=1, pred=1 ✓
  [0030] label=3, pred=3 ✓
  [0031] label=1, pred=1 ✓
  [0032] label=3, pred=3 ✓
  [0033] label=4, pred=4 ✓
  [0034] label=7, pred=7 ✓
  [0035] label=2, pred=2 ✓
  [0036] label=7, pred=7 ✓
 

'mlp_data'

In [16]:
import shutil
shutil.make_archive(OUTPUT_DIR, 'zip', '.', OUTPUT_DIR)
print(f'Created {OUTPUT_DIR}.zip — upload to PYNQ')

Created mlp_data.zip — upload to PYNQ
